In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import GPT2TokenizerFast
from datasets import load_dataset
from tqdm import tqdm


In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [14]:
dataset = load_dataset("roneneldan/TinyStories")

train_texts = dataset["train"]["text"][:10000]
test_texts = dataset["validation"]["text"][:2000]

In [15]:
train_texts[10]

'Once upon a time, there was a big car named Dependable. He had a very important job. Dependable would take a family to the park every day. The family had a mom, dad, and a little girl named Lily. They all had a lot of love for each other.\n\nOne day, when they got to the park, they saw a big sign that said, "Fun Race Today!" The family was very excited. They knew that Dependable was very fast and could win the race. So, they decided to join the race.\n\nThe race started, and Dependable went very fast. The other cars tried to catch up, but Dependable was too quick. In the end, Dependable won the race! The family was so happy and proud of their car. They knew that their love for each other and their trust in Dependable made them win the race. And from that day on, they had even more fun at the park, knowing that they had the fastest and most dependable car around.'

In [16]:
tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")

# GPT-2 has no pad token
tokenizer.pad_token = tokenizer.eos_token
vocab_size = tokenizer.vocab_size


In [17]:
class TinyStoriesDataset(Dataset):
  def __init__(self, texts, tokenizer, seq_len=64):
    self.seq_len = seq_len
    self.tokens = []

    for text in texts:
      ids = tokenizer.encode(text)
      self.tokens.extend(ids)
      # ids = tokenizer.encode(text) + [tokenizer.eos_token_id]

  def __len__(self):
    return len(self.tokens) - self.seq_len

  def __getitem__(self, idx):
    x = torch.tensor(self.tokens[idx:idx+self.seq_len])
    y = torch.tensor(self.tokens[idx+1:idx+self.seq_len+1])
    return x, y


In [18]:
seq_len = 4
batch_size = 128

train_ds = TinyStoriesDataset(train_texts, tokenizer, seq_len)
test_ds = TinyStoriesDataset(test_texts, tokenizer, seq_len)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=batch_size)

Token indices sequence length is longer than the specified maximum sequence length for this model (1106 > 1024). Running this sequence through the model will result in indexing errors


In [19]:
class RNNLanguageModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers, dropout):
        super().__init__()

        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.RNN(
            embed_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True
        )

        self.layer_norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden=None):
        x = self.embed(x)              # (B, T, E)
        out, hidden = self.rnn(x, hidden)
        out = self.layer_norm(out)     # normalization
        out = self.dropout(out)        # regularization
        logits = self.fc(out)          # (B, T, V)
        return logits, hidden


In [20]:
model = RNNLanguageModel(
    vocab_size=vocab_size,
    embed_dim=256,
    hidden_dim=512,
    num_layers=2,
    dropout=0.2
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()


In [21]:
def train_epoch(model, loader):
    model.train()
    total_loss = 0

    for x, y in tqdm(loader):
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        logits, _ = model(x)
        loss = criterion(
            logits.view(-1, vocab_size),
            y.view(-1)
        )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


In [22]:
def eval_epoch(model, loader):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits, _ = model(x)
            loss = criterion(
                logits.view(-1, vocab_size),
                y.view(-1)
            )
            total_loss += loss.item()

    return total_loss / len(loader)


In [24]:
import math
epochs = 2

for epoch in range(epochs):
    train_loss = train_epoch(model, train_loader)
    val_loss = eval_epoch(model, test_loader)

    print(f"Epoch {epoch+1}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss:   {val_loss:.4f}")
    print(f"Perplexity: {math.exp(val_loss):.2f}")
    print("-" * 40)

100%|██████████| 16814/16814 [11:29<00:00, 24.40it/s]


Epoch 1
Train Loss: 3.3949
Val Loss:   3.4168
Perplexity: 30.47
----------------------------------------


100%|██████████| 16814/16814 [11:24<00:00, 24.56it/s]


Epoch 2
Train Loss: 3.2141
Val Loss:   3.3692
Perplexity: 29.06
----------------------------------------


In [25]:
@torch.no_grad()
def generate_text(
    model,
    tokenizer,
    prompt,
    max_new_tokens=100,
    temperature=0.8
):
    model.eval()

    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    hidden = None

    for _ in range(max_new_tokens):
        logits, hidden = model(input_ids[:, -1:], hidden)
        logits = logits[:, -1, :] / temperature
        probs = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, 1)

        input_ids = torch.cat([input_ids, next_id], dim=1)

    return tokenizer.decode(input_ids[0])


In [26]:
prompts = [
    "Once upon a time",
    "The little cat",
    "In a small village",
]

for p in prompts:
    print("=" * 50)
    print(generate_text(model, tokenizer, p))


Once upon a time, there was a little girl named Lily. Lily was a little girl named Lily. She had to help her mommy. When she was feeling angry. They see the big and mean. You can be your dad."

"Let's play a game."

"My name is Emma. We love you for my turn?" Lily asks. "I want to catch it. It was so pretty and pretty!"

The people in the pond. The end.Once upon a time
The little cat said.

They took a few steps through the forest, there was a little girl named Jane. She was looking around the park. Everyone in the forest could see Lily.

"Hey Ben, what are you doing?" asked Jack and said, "I'm sorry, I will help you."

The ladybird thought it looked amazing. She says, "Please, why don't you try it too."

The moral of the story is: Be careful. He also
In a small village. She loved to play together at the gym and enjoyed his yummy food.

"Let's get the ball around the vine and go to the park and play with her mom and dad. They had fun. They played together so they all climbed into the b